# 02 — Feature Engineering
## FraudShield-ML

Transformamos los datos crudos en features listas para el modelo.

### Objetivos
- Limpiar columnas con alto porcentaje de nulos
- Crear features temporales desde TransactionDT
- Definir el split estratificado train/val/test (70/15/15)
- Construir el ColumnTransformer con imputacion, scaling y encoding
- Guardar conjuntos procesados y el preprocesador


## 1. Importaciones

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder

import joblib

print("Librerías cargadas")

Librerías cargadas


## 2. Carga del dataset

In [2]:
PROJECT_ROOT   = Path(".").resolve().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

train = pd.read_parquet(DATA_PROCESSED / "train_merged.parquet")
test  = pd.read_parquet(DATA_PROCESSED / "test_merged.parquet")

print(f"train: {train.shape}")
print(f"test:  {test.shape}")

train: (590540, 434)
test:  (506691, 433)


## 3. Primera vista

In [3]:
train.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


## 4. Limpieza de nombres de columnas y nulos

Reemplazamos guiones por guiones bajos y eliminamos columnas con más del 90% de nulos.

In [4]:
# Reemplazar guiones por guiones bajos en nombres de columnas
train.columns = train.columns.str.replace('-', '_', regex=False)
test.columns  = test.columns.str.replace('-', '_', regex=False)

# Verificar que no quedaron guiones
problemas_train = [c for c in train.columns if '-' in c]
problemas_test  = [c for c in test.columns  if '-' in c]

print(f"Columnas con guión en train: {len(problemas_train)}")
print(f"Columnas con guión en test:  {len(problemas_test)}")

Columnas con guión en train: 0
Columnas con guión en test:  0


## 5. Análisis de valores faltantes

In [5]:
# Calcular porcentaje de nulos por columna
pct_nulos = train.isnull().sum() / len(train) * 100

# Identificar columnas a eliminar
umbral    = 90
cols_eliminar = pct_nulos[pct_nulos > umbral].index.tolist()

print(f"Umbral:              {umbral}%")
print(f"Columnas eliminadas: {len(cols_eliminar)}")
print(f"Nombres:             {cols_eliminar}")

train = train.drop(columns=cols_eliminar)
test  = test.drop(columns=cols_eliminar, errors='ignore')

print(f"\ntrain después de eliminar: {train.shape}")
print(f"test después de eliminar:  {test.shape}")

Umbral:              90%
Columnas eliminadas: 12
Nombres:             ['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']

train después de eliminar: (590540, 422)
test después de eliminar:  (506691, 421)


## 6. Feature Engineering — Features temporales

Extraemos hora del dia, dia de la semana y transformacion logaritmica del monto.
Estas features capturan patrones de comportamiento que el modelo puede aprender.

In [6]:
# Hora del día y día de la semana desde TransactionDT
train['hora']       = (train['TransactionDT'] // 3600) % 24
train['dia_semana'] = (train['TransactionDT'] // 86400) % 7
train['log_amt']    = np.log1p(train['TransactionAmt'])

test['hora']        = (test['TransactionDT'] // 3600) % 24
test['dia_semana']  = (test['TransactionDT'] // 86400) % 7
test['log_amt']     = np.log1p(test['TransactionAmt'])

print(f"hora       — rango: {train['hora'].min()} a {train['hora'].max()}")
print(f"dia_semana — rango: {train['dia_semana'].min()} a {train['dia_semana'].max()}")
print(f"log_amt    — rango: {train['log_amt'].min():.2f} a {train['log_amt'].max():.2f}")
print(f"\ntrain shape: {train.shape}")

hora       — rango: 0 a 23
dia_semana — rango: 0 a 6
log_amt    — rango: 0.22 a 10.37

train shape: (590540, 425)


## 7. Separar features y target

In [7]:
# Columnas que no son features
cols_excluir = ['isFraud', 'TransactionID', 'TransactionDT']

X = train.drop(columns=cols_excluir)
y = train['isFraud']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"% fraude: {y.mean()*100:.2f}%")

X shape: (590540, 422)
y shape: (590540,)
% fraude: 3.50%


## 8. Split estratificado train / val / test

Usamos split estratificado para preservar el desbalance 96.5/3.5 en los tres conjuntos.

In [8]:
# Primer split: 70% train, 30% temporal
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Segundo split: el 30% temporal se divide en 15% val y 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(f"X_train: {X_train.shape}  |  fraude: {y_train.mean()*100:.2f}%")
print(f"X_val:   {X_val.shape}  |  fraude: {y_val.mean()*100:.2f}%")
print(f"X_test:  {X_test.shape}  |  fraude: {y_test.mean()*100:.2f}%")

X_train: (413378, 422)  |  fraude: 3.50%
X_val:   (88581, 422)  |  fraude: 3.50%
X_test:  (88581, 422)  |  fraude: 3.50%


## 9. Identificar columnas por tipo

In [9]:
# Columnas numéricas
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Columnas categóricas
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Separar categóricas por cardinalidad
umbral_cardinalidad = 10
cat_low  = [c for c in cat_cols if X_train[c].nunique() <= umbral_cardinalidad]
cat_high = [c for c in cat_cols if X_train[c].nunique() >  umbral_cardinalidad]

print(f"Columnas numéricas:            {len(num_cols)}")
print(f"Categóricas baja cardinalidad: {len(cat_low)}  → OrdinalEncoder")
print(f"Categóricas alta cardinalidad: {len(cat_high)} → TargetEncoder")

print("\nCategóricas baja cardinalidad:")
for c in cat_low:
    print(f"  {c} — {X_train[c].nunique()} valores únicos")

print("\nCategóricas alta cardinalidad:")
for c in cat_high:
    print(f"  {c} — {X_train[c].nunique()} valores únicos")

Columnas numéricas:            393
Categóricas baja cardinalidad: 23  → OrdinalEncoder
Categóricas alta cardinalidad: 6 → TargetEncoder

Categóricas baja cardinalidad:
  ProductCD — 5 valores únicos
  card4 — 4 valores únicos
  card6 — 4 valores únicos
  M1 — 2 valores únicos
  M2 — 2 valores únicos
  M3 — 2 valores únicos
  M4 — 3 valores únicos
  M5 — 2 valores únicos
  M6 — 2 valores únicos
  M7 — 2 valores únicos
  M8 — 2 valores únicos
  M9 — 2 valores únicos
  id_12 — 2 valores únicos
  id_15 — 3 valores únicos
  id_16 — 2 valores únicos
  id_28 — 2 valores únicos
  id_29 — 2 valores únicos
  id_34 — 4 valores únicos
  id_35 — 2 valores únicos
  id_36 — 2 valores únicos
  id_37 — 2 valores únicos
  id_38 — 2 valores únicos
  DeviceType — 2 valores únicos

Categóricas alta cardinalidad:
  P_emaildomain — 59 valores únicos
  R_emaildomain — 60 valores únicos
  id_30 — 75 valores únicos
  id_31 — 122 valores únicos
  id_33 — 226 valores únicos
  DeviceInfo — 1619 valores únicos


## 10. Construccion del ColumnTransformer

Definimos pipelines separados para columnas numericas y categoricas.
El TargetEncoder se ajusta solo con datos de entrenamiento para evitar leakage.

In [10]:
# Rama numérica: imputar con mediana y escalar
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Rama categórica baja cardinalidad: imputar con moda y encoding ordinal
low_card_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Rama categórica alta cardinalidad: imputar con moda y target encoding
high_card_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder())
])

# Combinar las tres ramas
preprocessor = ColumnTransformer(transformers=[
    ('num',      numeric_transformer,   num_cols),
    ('cat_low',  low_card_transformer,  cat_low),
    ('cat_high', high_card_transformer, cat_high)
])

print("Pipeline construido")
print(f"  Rama numérica:    {len(num_cols)} columnas")
print(f"  Rama cat_low:     {len(cat_low)} columnas")
print(f"  Rama cat_high:    {len(cat_high)} columnas")

Pipeline construido
  Rama numérica:    393 columnas
  Rama cat_low:     23 columnas
  Rama cat_high:    6 columnas


## 11. Ajuste y transformacion del preprocesador

In [11]:
# Ajustar SOLO con X_train
# TargetEncoder necesita y_train para calcular tasas de fraude por categoría
preprocessor.fit(X_train, y_train)

# Transformar los tres conjuntos
X_train_prep = preprocessor.transform(X_train)
X_val_prep   = preprocessor.transform(X_val)
X_test_prep  = preprocessor.transform(X_test)

print(f"X_train_prep: {X_train_prep.shape}")
print(f"X_val_prep:   {X_val_prep.shape}")
print(f"X_test_prep:  {X_test_prep.shape}")

X_train_prep: (413378, 422)
X_val_prep:   (88581, 422)
X_test_prep:  (88581, 422)


## 12. Guardar conjuntos procesados y preprocesador

In [12]:
PROJECT_ROOT   = Path(".").resolve().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR     = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Guardar conjuntos procesados
pd.DataFrame(X_train_prep).to_parquet(DATA_PROCESSED / "X_train.parquet", index=False)
pd.DataFrame(X_val_prep).to_parquet(DATA_PROCESSED / "X_val.parquet",     index=False)
pd.DataFrame(X_test_prep).to_parquet(DATA_PROCESSED / "X_test.parquet",   index=False)

# Guardar targets
y_train.to_frame().to_parquet(DATA_PROCESSED / "y_train.parquet", index=False)
y_val.to_frame().to_parquet(DATA_PROCESSED / "y_val.parquet",     index=False)
y_test.to_frame().to_parquet(DATA_PROCESSED / "y_test.parquet",   index=False)

# Guardar preprocesador
joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

print("Datos procesados guardados en data/processed/")
print("Preprocesador guardado en models/preprocessor.joblib")

Datos procesados guardados en data/processed/
Preprocesador guardado en models/preprocessor.joblib
